# Exemplo 11 - RH: um assistente que responde pelo manual da empresa (RAG)

### Palestra Machine Learning e IA: Da máquina que aprende à IA que age

### Prof. Dr. Ahirton Lopes (https://github.com/ahirtonlopes)

**Onde isso aparece no mercado:** Assistentes internos de RH, jurídico e TI, chatbots de atendimento que respondem com base na base de conhecimento da empresa e buscadores de documentos com resposta em linguagem natural. Quase todo "ChatGPT da empresa" que você vai encontrar no mercado é, por baixo, um RAG.

**Conceito da palestra:** RAG (*Retrieval-Augmented Generation*, geração aumentada por recuperação): antes de responder, o sistema **busca** os trechos certos numa base própria e entrega esses trechos ao LLM junto com a pergunta. O modelo não é re-treinado; ele só passa a "ler a cola certa" antes de responder.

> As Partes 1 e 2 rodam **sem nenhuma chave de API**. As Partes 3 a 5 usam o Gemini e precisam de uma API key gratuita do Google AI Studio (o passo a passo está no README do repositório, na seção do exemplo 10).

## Por que RAG?

Um LLM sozinho tem dois limites importantes:

1. **Ele só sabe o que viu no treino.** Não conhece o manual da sua empresa, o contrato do seu cliente nem a política que mudou ontem.
2. **Ele pode inventar** (a famosa *alucinação*): quando não sabe, às vezes responde com confiança mesmo assim.

O RAG ataca os dois problemas com uma receita de 3 passos:

```
Pergunta ──► 1. BUSCAR os trechos mais parecidos na base própria
                    │
                    ▼
             2. MONTAR um prompt: "responda usando SÓ estes trechos"
                    │
                    ▼
             3. GERAR a resposta com o LLM, citando a fonte
```

Neste notebook vamos montar isso do zero, em cima do manual do colaborador de uma empresa **fictícia** de São Paulo: a **Garoa Tecnologia**.

## Parte 0: Setup

In [1]:
!pip install -q google-genai scikit-learn

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
print("✅ Bibliotecas prontas!")

✅ Bibliotecas prontas!


## Parte 1: A base de conhecimento

Em uma empresa real, esta base viria de PDFs, páginas da intranet ou de um sistema de tickets, quebrados em pedaços menores (os *chunks*). Aqui, para ficar simples, o manual já está dividido em 12 trechos, um por política.

In [2]:
manual = [
    {"secao": "Trabalho híbrido",
     "texto": "Na Garoa Tecnologia o modelo de trabalho é híbrido: 3 dias por semana no escritório da Avenida Paulista "
              "(terça, quarta e quinta) e 2 dias em home office (segunda e sexta). Times de suporte seguem escala própria definida pela liderança."},
    {"secao": "Jornada e horário",
     "texto": "A jornada é flexível entre 7h e 20h, com núcleo obrigatório das 10h às 16h para reuniões. "
              "Horas extras entram no banco de horas e devem ser compensadas em até 90 dias."},
    {"secao": "Vale-refeição e alimentação",
     "texto": "O benefício de refeição e alimentação é de R$ 55 por dia útil, creditado no cartão até o 5º dia útil do mês. "
              "O colaborador pode dividir o saldo entre refeição e alimentação pelo aplicativo do benefício."},
    {"secao": "Transporte e mobilidade",
     "texto": "A empresa oferece vale-transporte ou, para quem vem de bicicleta, patinete ou carona, auxílio-mobilidade de R$ 250 por mês. "
              "Corridas de aplicativo só são reembolsadas depois das 22h ou em visitas a clientes, com nota fiscal lançada no sistema de despesas em até 30 dias."},
    {"secao": "Férias",
     "texto": "São 30 dias de férias por ano, que podem ser divididos em até 3 períodos, sendo um deles de pelo menos 14 dias. "
              "O pedido deve ser feito no portal de RH com 45 dias de antecedência. Pela CLT, as férias não podem começar nos 2 dias que antecedem feriado ou descanso semanal."},
    {"secao": "Day off de aniversário",
     "texto": "Todo colaborador tem uma folga no dia do aniversário ou em qualquer outro dia útil do mesmo mês, combinada com a liderança."},
    {"secao": "Plano de saúde",
     "texto": "Plano de saúde e odontológico sem custo para o colaborador. Dependentes (cônjuge e filhos) têm coparticipação de 20% nas consultas. "
              "A inclusão de dependentes deve ser pedida em até 30 dias após a admissão, o casamento ou o nascimento."},
    {"secao": "Equipamentos",
     "texto": "O notebook é entregue no primeiro dia. Há uma ajuda de custo única de R$ 1.200 para montar a estação de trabalho em casa "
              "(cadeira, monitor, teclado), mediante nota fiscal enviada em até 60 dias."},
    {"secao": "Educação e certificações",
     "texto": "Após 3 meses de empresa, o colaborador tem auxílio-educação de até R$ 600 por mês para graduação, pós-graduação ou idiomas. "
              "Certificações em nuvem e IA são reembolsadas integralmente após a aprovação no exame."},
    {"secao": "Uso de IA generativa",
     "texto": "Ferramentas de IA generativa aprovadas pela TI podem ser usadas no dia a dia. É proibido colar dados de clientes, código proprietário "
              "ou dados pessoais em ferramentas não aprovadas. Todo conteúdo gerado por IA deve ser revisado por uma pessoa antes de ser enviado ao cliente."},
    {"secao": "Licença-maternidade e paternidade",
     "texto": "A Garoa participa do programa Empresa Cidadã: a licença-maternidade é de 180 dias e a licença-paternidade é de 20 dias."},
    {"secao": "Canal de ética",
     "texto": "Denúncias podem ser feitas de forma anônima pelo canal de ética, disponível 24 horas e operado por uma empresa externa. "
              "Qualquer retaliação contra quem denuncia é proibida."},
]

df = pd.DataFrame(manual)
print(f"📚 {len(df)} trechos no manual da Garoa Tecnologia")
df

📚 12 trechos no manual da Garoa Tecnologia


,secao,texto
0,Trabalho híbrido,"Na Garoa Tecnologia o modelo de trabalho é híbrido: 3 dias por semana no escritório da Avenida Paulista (terça, quar..."
1,Jornada e horário,"A jornada é flexível entre 7h e 20h, com núcleo obrigatório das 10h às 16h para reuniões. Horas extras entram no ban..."
2,Vale-refeição e alimentação,"O benefício de refeição e alimentação é de R$ 55 por dia útil, creditado no cartão até o 5º dia útil do mês. O colab..."
3,Transporte e mobilidade,"A empresa oferece vale-transporte ou, para quem vem de bicicleta, patinete ou carona, auxílio-mobilidade de R$ 250 p..."
4,Férias,"São 30 dias de férias por ano, que podem ser divididos em até 3 períodos, sendo um deles de pelo menos 14 dias. O pe..."
5,Day off de aniversário,"Todo colaborador tem uma folga no dia do aniversário ou em qualquer outro dia útil do mesmo mês, combinada com a lid..."
6,Plano de saúde,Plano de saúde e odontológico sem custo para o colaborador. Dependentes (cônjuge e filhos) têm coparticipação de 20%...
7,Equipamentos,O notebook é entregue no primeiro dia. Há uma ajuda de custo única de R$ 1.200 para montar a estação de trabalho em ...
8,Educação e certificações,"Após 3 meses de empresa, o colaborador tem auxílio-educação de até R$ 600 por mês para graduação, pós-graduação ou i..."
9,Uso de IA generativa,"Ferramentas de IA generativa aprovadas pela TI podem ser usadas no dia a dia. É proibido colar dados de clientes, có..."


## Parte 2: Buscar (o "R" do RAG), primeiro do jeito clássico

O passo mais importante do RAG é a **busca**. Se a busca traz o trecho errado, o LLM responde errado, por mais inteligente que seja.

Vamos começar com uma busca clássica, a mesma ideia do **TF-IDF** que usamos no exemplo 01 para ler URLs: cada texto vira um vetor de **palavras**, e o trecho mais parecido com a pergunta é o que tem mais palavras importantes em comum com ela.

In [3]:
tfidf = TfidfVectorizer()
matriz_manual = tfidf.fit_transform(df["texto"])

def buscar_tfidf(pergunta, k=3):
    """Retorna os k trechos com mais palavras em comum com a pergunta."""
    vetor_pergunta = tfidf.transform([pergunta])
    scores = cosine_similarity(vetor_pergunta, matriz_manual)[0]
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({"secao": df["secao"].iloc[top].values,
                         "similaridade": scores[top].round(3)})

buscar_tfidf("Qual o valor do vale-refeição por dia?")

,secao,similaridade
0,Vale-refeição e alimentação,0.434
1,Day off de aniversário,0.291
2,Transporte e mobilidade,0.122


Funcionou: a pergunta usa as **mesmas palavras** do manual ("vale-refeição", "dia"). Agora veja o que acontece quando a pessoa pergunta do jeito que fala no dia a dia:

In [4]:
perguntas_do_dia_a_dia = [
    "Posso trabalhar de casa na segunda-feira?",
    "Meu Uber na volta do happy hour é reembolsado?",
    "Vou ser pai em março, quantos dias posso ficar em casa?",
]

for p in perguntas_do_dia_a_dia:
    print(f"❓ {p}")
    print(buscar_tfidf(p, k=2).to_string(index=False))
    print()

❓ Posso trabalhar de casa na segunda-feira?
           secao  similaridade
Trabalho híbrido         0.248
    Equipamentos         0.167

❓ Meu Uber na volta do happy hour é reembolsado?
                      secao  similaridade
     Day off de aniversário         0.234
Vale-refeição e alimentação         0.166

❓ Vou ser pai em março, quantos dias posso ficar em casa?
       secao  similaridade
Equipamentos         0.273
      Férias         0.211



Repare nos resultados acima:

- **Trabalho em casa:** acertou, mas por sorte: a pergunta repete "segunda", palavra que aparece no trecho do trabalho híbrido.
- **Uber:** errou. O manual fala em "corridas de aplicativo", nunca em "Uber", e o trecho certo (Transporte e mobilidade) nem aparece entre os dois primeiros.
- **Ser pai:** errou. A palavra "casa" puxou o trecho de Equipamentos ("estação de trabalho em casa"), e "licença-paternidade" nem foi encontrada.

A busca por palavras se confunde com **sinônimos** e com o jeito como as pessoas realmente perguntam.

É aqui que entram os **embeddings**: vetores que representam o **significado** do texto, não as palavras exatas. Textos que querem dizer a mesma coisa ficam próximos, mesmo sem nenhuma palavra em comum.

> A partir daqui você precisa da sua API key do Google AI Studio.

## Parte 3: Configurar o Gemini 🔑

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key carregada dos Secrets do Colab.")
except Exception:
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Cole sua API key do Google AI Studio: ")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Modelos disponíveis para a sua chave
modelos = list(client.models.list())
modelos_embedding = [m.name.removeprefix("models/") for m in modelos if "embedContent" in (m.supported_actions or [])]
print("Modelos de embedding disponíveis:", modelos_embedding)

# 🎯 Modelos usados neste notebook (troque aqui se algum não aparecer na sua lista)
MODELO_EMBEDDING = "gemini-embedding-001" if "gemini-embedding-001" in modelos_embedding else modelos_embedding[0]
MODELO_LLM = "gemini-3.1-flash-lite"   # o mesmo do exemplo 10, com cota maior no plano gratuito
print(f"\n✅ Embedding: {MODELO_EMBEDDING} | LLM: {MODELO_LLM}")

## Parte 4: Buscar por significado com embeddings

Vamos transformar cada trecho do manual em um vetor de significado (uma vez só) e, a cada pergunta, transformar a pergunta também. O trecho mais próximo da pergunta é o mais relevante.

In [ ]:
def embed(textos, tarefa):
    resp = client.models.embed_content(
        model=MODELO_EMBEDDING,
        contents=textos,
        config=types.EmbedContentConfig(task_type=tarefa),
    )
    vetores = np.array([e.values for e in resp.embeddings])
    return vetores / np.linalg.norm(vetores, axis=1, keepdims=True)  # normaliza para usar produto escalar

vetores_manual = embed(df["texto"].tolist(), "RETRIEVAL_DOCUMENT")
print(f"✅ {vetores_manual.shape[0]} trechos viraram vetores de {vetores_manual.shape[1]} números cada.")

def buscar_semantico(pergunta, k=3):
    v = embed([pergunta], "RETRIEVAL_QUERY")[0]
    scores = vetores_manual @ v
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({"secao": df["secao"].iloc[top].values,
                         "similaridade": scores[top].round(3),
                         "texto": df["texto"].iloc[top].values})

In [ ]:
for p in perguntas_do_dia_a_dia:
    print(f"❓ {p}")
    print("   Por palavras (TF-IDF): ", buscar_tfidf(p, k=1)["secao"].iloc[0])
    print("   Por significado:       ", buscar_semantico(p, k=1)["secao"].iloc[0])
    print()

## Parte 5: Gerar a resposta (o "AG" do RAG)

Agora juntamos tudo: buscamos os 3 trechos mais relevantes, colocamos no prompt e pedimos ao LLM que responda **usando apenas esses trechos** e citando a seção. Se a resposta não estiver lá, ele deve dizer que não sabe, em vez de inventar.

In [ ]:
INSTRUCOES = """Você é o assistente de RH da Garoa Tecnologia.
Responda em português, de forma curta e simpática, usando APENAS os trechos do manual fornecidos.
Ao final, cite entre colchetes a(s) seção(ões) usada(s), por exemplo: [Férias].
Se a resposta não estiver nos trechos, diga que não encontrou essa informação no manual e sugira procurar o time de RH. Nunca invente."""

def perguntar(pergunta, k=3, mostrar_contexto=False):
    trechos = buscar_semantico(pergunta, k=k)
    contexto = "\n\n".join(f"[{s}] {t}" for s, t in zip(trechos["secao"], trechos["texto"]))
    prompt = f"Trechos do manual:\n{contexto}\n\nPergunta: {pergunta}"
    if mostrar_contexto:
        print("📎 Contexto enviado ao modelo:\n" + contexto + "\n")
    resposta = client.models.generate_content(
        model=MODELO_LLM,
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=INSTRUCOES, temperature=0.2),
    )
    return resposta.text

print(perguntar("Posso trabalhar de casa na segunda-feira?", mostrar_contexto=True))

In [ ]:
for p in ["Meu Uber na volta do happy hour é reembolsado?",
          "Vou ser pai em março, quantos dias posso ficar em casa?",
          "Posso usar o ChatGPT pra resumir a planilha de clientes?"]:
    print(f"❓ {p}\n💬 {perguntar(p)}\n")

### O teste mais importante: uma pergunta que o manual **não** responde

Um bom assistente sabe dizer "não sei". Repare que a resposta não deve inventar um valor.

In [ ]:
print(perguntar("Qual é o salário de um analista de dados júnior na Garoa?"))

### E sem RAG?

Para comparar, vamos fazer uma pergunta ao mesmo modelo **sem** entregar os trechos do manual. Ele não conhece a Garoa Tecnologia (ela nem existe!), então qualquer número específico que aparecer foi tirado de padrões gerais, não da política real da empresa.

In [ ]:
sem_rag = client.models.generate_content(
    model=MODELO_LLM,
    contents="Na Garoa Tecnologia, quanto é o vale-refeição por dia e quantos dias de licença-paternidade eu tenho?",
)
print(sem_rag.text)

## 🧪 Agora é com você

1. **Troque a base:** substitua a lista `manual` por trechos de um documento da sua área (regulamento da faculdade, edital de estágio, FAQ de um produto) e faça perguntas.
2. **Mude o `k`:** o que acontece com as respostas se o modelo receber só 1 trecho? E 6?
3. **Quebre o assistente:** tente perguntas ambíguas ou que misturam duas políticas ("posso emendar o day off com as férias?"). Onde a busca falha?
4. **Compare as buscas:** crie suas próprias perguntas com sinônimos e veja quando o TF-IDF acerta e quando só a busca por significado resolve.

> 💡 **Para ir além:** em produção, os vetores ficam guardados em um **banco vetorial** (como Vertex AI Vector Search, pgvector ou Chroma), os documentos são quebrados em pedaços automaticamente e a qualidade das respostas é medida com conjuntos de perguntas de teste. A ideia central, porém, é exatamente a que você acabou de montar.